### Setup & Imports

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/devla.fwilliams/psav-payment-poc")
SRC_DIR = PROJECT_ROOT / "src"
DATA_DIR = PROJECT_ROOT / "data" / "staging"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from data_cleaning import clean_dataset
from feature_engineering import create_features

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

### Load Dataset

In [7]:
df = pd.read_parquet(
    DATA_DIR / "psav_model_2025-01-01_to_2026-01-01_pulled_2026-03-06.parquet",
    engine="pyarrow"
)

df.shape

(1000000, 124)

### Clean and Engineer EDA features

In [ ]:
data = clean_dataset(df)
data = create_features(data)

print("Shape:", data.shape)
data.head()

Shape: (1000000, 31)


,client_code,savings,totalcharges,parent_client_code,networkcode,claim_type,cpt,claim_fail_reason,clientname,noncontractcode,noncoveredcharges,placeofservice,providergroupname,reasonableandcustomary,totalclaimdays,provider_state,mpi_surprise_bill_ind,description,speciality_category,speciality_category_desc,days_since_received,billed_amount_sum,paid_flag,zz1_supergroupid_op1,name_org,zz1_clientcode_bh_op1,zz1_acctproduct_op1,zz1_product_group_id_op1,zz1_mda_product_id_op1,zz1_solution_id_op1,blart
0,USNAS,158.00,790.00,USNAS-P,MPFN,HCFA,80307,ZL,UMR USNAS,2,0.0,21,ANCILLARY PATHWAYSLLC,0.00,0,FL,X,UMR USNAS,Pathology and Laboratory,DRUG TST PRSMV INSTRMNT CHEM ANALYZERS PR DAT,289,8.69,0,1506,UMR USNASUMR USNASUMR USNAS,USNAS,FNX,PG070,MP012,SO001,ZS
1,HSP CONSPBNS,10537.00,11500.00,HSP-P,DataISightFac,UB,H0035,DIS,Cigna East,2,0.0,10,INSPIRE RECOVERY LLC,11500.00,0,FL,X,Cigna East,Behavioral Health Treatments/Services,Partial hospitalization-Intensive,210,895.65,0,3628,Cigna East CIGNA NSP & BNS,HSP CONSPBNS,DATAISIGHTFAC,PG035,MP008,SO001,ZS
2,HSP CONSPBNS,20475.00,62400.00,HSP-P,MPFac,UB,H0035,B01,Cigna East,-1,0.0,10,WILLOWS AT RED OAK RECOV,62400.00,0,NC,X,Cigna East,Behavioral Health Treatments/Services,"PARTIAL HOSPITALIZATION, LESS INTENSIVE",90,1638.00,0,3628,Cigna East CIGNA NSP & BNS,HSP CONSPBNS,FACILITY,PG062,MP013,SO005,ZS
3,HSP CONSPBNS,80.08,429.00,HSP-P,MPPract,HCFA,96127,A1,Cigna East,-1,0.0,11,"BALA III MD, JUAN F",406.00,0,WA,N,Cigna East,Evaluation and Management,OFFICE/OUTPATIENT ESTABLISHED MOD MDM 30-39 M,288,6.41,0,3628,Cigna East CIGNA NSP & BNS,HSP CONSPBNS,PRACTITIONER,PG062,MP013,SO005,ZS
4,CBSA DIS,1454.05,1915.09,CBSA-P,DataISightPract,HCFA,A0425,DIP,Meritain - Minneapolis,2,0.0,41,GOLD CROSS AMBULANCE,1742.59,0,UT,X,Meritain - Minneapolis,Ground Ambulance,ALS 1,280,174.49,0,3249,Meritain - Minneapolis DIS ONLYMeritain - Minn...,CBSA DIS,DATAISIGHTPRACT,PG035,MP008,SO001,ZS


### Overview

In [ ]:
data.info()

### Target Check

In [ ]:
print(data["paid_flag"].value_counts())
print(data["paid_flag"].value_counts(normalize=True))

sns.countplot(data=data, x="paid_flag")
plt.title("Paid Flag Distribution")
plt.tight_layout()
plt.savefig(FIG_DIR / "target_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

,zz1_acctproduct_op1,zz1_product_group_id_op1,zz1_mda_product_id_op1,zz1_solution_id_op1
0,FNX,PG070,MP012,SO001
1,DATAISIGHTFAC,PG035,MP008,SO001
2,FACILITY,PG062,MP013,SO005
3,PRACTITIONER,PG062,MP013,SO005
4,DATAISIGHTPRACT,PG035,MP008,SO001
5,XCLINICALREVIEW,PG027,MP005,SO006
6,MPGLOBALS,PG066,MP013,SO005
7,PRACTITIONER,PG062,MP013,SO005
8,FNX,PG070,MP012,SO001
9,NSA_QPABASDPRCNG,PG086,MP028,SO001


### Missingness

In [ ]:
missing = data.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
display(missing)

if not missing.empty:
    missing.plot(kind="bar", figsize=(10, 4))
    plt.title("Missing Values by Feature")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "missing_values.png", dpi=150, bbox_inches="tight")
    plt.show()

### Split Feature Types

In [ ]:
numeric_cols = data.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
cat_cols = data.select_dtypes(include=["object", "string", "category"]).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(cat_cols))
print("\nNumeric:", numeric_cols)
print("\nCategorical:", cat_cols)

### Numeric Summary

In [ ]:
display(data[numeric_cols].describe().T)
display(data[numeric_cols].quantile([0.95, 0.99, 0.999]).T)

### Cardinality and Top Values

In [ ]:
cardinality = data[cat_cols].nunique().sort_values(ascending=False)
display(cardinality)

for col in cat_cols:
    print(f"\nTop values for {col}")
    display(data[col].value_counts(dropna=False).head(10))

### Correlation Heatmap

In [ ]:
corr = data[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(14, 10))
sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.5)
plt.title("Numeric Correlation Matrix")
plt.tight_layout()
plt.savefig(FIG_DIR / "correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

corr_with_target = data[numeric_cols].corrwith(data["paid_flag"]).sort_values(ascending=False)
display(corr_with_target)

### Numeric VS Target

In [ ]:
plot_cols = [
    "billed_amount_sum",
    "reasonableandcustomary",
    "noncoveredcharges",
    "days_since_received",
    "totalcharges",
    "savings",
    "rnc_to_billed_ratio",
    "noncovered_to_billed_ratio",
]

plot_cols = [c for c in plot_cols if c in data.columns]

for col in plot_cols:
    plt.figure(figsize=(6, 4))
    sns.boxplot(data=data, x="paid_flag", y=col)
    plt.title(f"{col} vs paid_flag")
    plt.tight_layout()
    plt.show()

### Log Distributions and Categorical VS Target

In [ ]:
log_plot_cols = [
    "log_billed_amount_sum",
    "log_reasonableandcustomary",
    "log_noncoveredcharges",
]

for col in [c for c in log_plot_cols if c in data.columns]:
    plt.figure(figsize=(7, 4))
    sns.kdeplot(data=data, x=col, hue="paid_flag", fill=True, common_norm=False)
    plt.title(f"{col} by paid_flag")
    plt.tight_layout()
    plt.show()

for col in [c for c in ["claim_type", "provider_state", "zz1_acctproduct_op1", "claim_age_bucket"] if c in data.columns]:
    summary = (
        data.groupby(col, dropna=False)["paid_flag"]
        .agg(["mean", "count"])
        .sort_values("count", ascending=False)
    )
    print(f"\nTarget rate summary for {col}")
    display(summary.head(20))